In [1]:
import pandas as pd
import codecs
from google.colab import drive

In [2]:
#importando do drive
drive.mount('/content/drive')

# colocar o caminho no drive. Por hora estou usando o meu local
caminho_arquivo = "/content/drive/MyDrive/UFRN 2024/2026/BNsql/ABRaOM_60+_SABE_609_exomes_annotated.txt"

# Testando os encodings (via gemini)
encodings_para_testar = ['utf-8', 'iso-8859-1', 'cp1252']
encoding_detectado = None

for enc in encodings_para_testar:
    try:
        with codecs.open(caminho_arquivo, "r", encoding=enc) as f:
            f.read(1024) # Testa a leitura dos primeiros bytes
        encoding_detectado = enc
        print(f"[OK] Codificação detectada: {encoding_detectado}")
        break
    except UnicodeDecodeError:
        continue

if not encoding_detectado:
    encoding_detectado = 'utf-8' # Fallback caso mude algo
    print("[Aviso] Não foi possível cravar a codificação. Usando utf-8 por padrão.")

separador = "\t"

# Quantidade de colunas e registros
with open(caminho_arquivo, 'r', encoding=encoding_detectado) as f:
    total_linhas = sum(1 for linha in f)

total_registros = total_linhas - 1

df_colunas = pd.read_csv(caminho_arquivo, sep=separador, nrows=1, encoding=encoding_detectado)
total_colunas = len(df_colunas.columns)

print(f"Quantidade de colunas: {total_colunas}")
print(f"Quantidade de registros: {total_registros}")

# VERIFICAR TIPOS DE DADOS
df_amostra = pd.read_csv(caminho_arquivo, sep=separador, nrows=1000, encoding=encoding_detectado)
print("\nTipos de dados estimados por coluna:")
print(df_amostra.dtypes)

#CARREGAR O DATAFRAME BRUTO
df_bruto = pd.read_csv(caminho_arquivo, sep=separador, encoding=encoding_detectado)
print("DataFrame 'df_bruto' criado!")



Mounted at /content/drive
[OK] Codificação detectada: utf-8
Quantidade de colunas: 16
Quantidade de registros: 2382573

Tipos de dados estimados por coluna:
Chr                          int64
Start                        int64
Ref                         object
Alt                         object
PredictedFunc.refGene       object
Gene.refGene                object
PredConsequence.refGene     object
avsnp147                    object
FILTER                      object
CEGH Filter                 object
HomozygousALT count          int64
Hemizygous count             int64
Allele number                int64
Allele ALT count             int64
Frequencies                float64
Cohort                      object
dtype: object


/tmp/ipykernel_48160/669636433.py:45: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_bruto = pd.read_csv(caminho_arquivo, sep=separador, encoding=encoding_detectado)


DataFrame 'df_bruto' criado!


In [4]:
#Contar variantes
total_variantes = len(df_bruto)

# Cont genes únicos (coluna Gene.refGene)
genes_unicos = df_bruto['Gene.refGene'].nunique()

#quantidade de valores nulos por coluna
valores_nulos = df_bruto.isnull().sum()

#  distribuição das colunas categóricas solicitadas (gemini)
dist_func = df_bruto['PredictedFunc.refGene'].value_counts(dropna=False)
dist_consequence = df_bruto['PredConsequence.refGene'].value_counts(dropna=False)
dist_cohort = df_bruto['Cohort'].value_counts(dropna=False)

#(gemini)
# --- EXIBIÇÃO DO RELATÓRIO SIMPLES DE QUALIDADE DOS DADOS ---
print("\n" + "="*40)
print("  RELATÓRIO DE QUALIDADE DOS DADOS (QC)")
print("="*40)
print(f"• Total de Variantes (Registros): {total_variantes}")
print(f"• Total de Genes Únicos: {genes_unicos}")

print("\n• Valores Nulos Identificados por Coluna:")
#colunas nulas
colunas_com_nulos = valores_nulos[valores_nulos > 0]
if len(colunas_com_nulos) > 0:
    print(colunas_com_nulos)
else:
    print("Nenhum valor nulo (NaN) estrito foi encontrado no arquivo!")

print("\n• Distribuição: PredictedFunc.refGene")
print(dist_func)

print("\n• Distribuição: PredConsequence.refGene")
print(dist_consequence)

print("\n• Distribuição: Cohort")
print(dist_cohort)
print("="*40)


  RELATÓRIO DE QUALIDADE DOS DADOS (QC)
• Total de Variantes (Registros): 2382573
• Total de Genes Únicos: 28127

• Valores Nulos Identificados por Coluna:
PredConsequence.refGene    1657011
avsnp147                    669289
CEGH Filter                   1621
dtype: int64

• Distribuição: PredictedFunc.refGene
PredictedFunc.refGene
intronic                   1352308
UTR3                        224416
nonsynonymous SNV           224353
synonymous SNV              162746
ncRNA_intronic              101136
UTR5                         76981
upstream                     75677
ncRNA_exonic                 71226
downstream                   51209
intergenic                   11303
unknown                       5100
upstream;downstream           4871
nonframeshift deletion        4478
stopgain                      4066
frameshift deletion           3489
frameshift insertion          3011
splicing                      3000
nonframeshift insertion       2279
ncRNA_splicing                 398

In [5]:
#colunas para manter
colunas_para_manter = [
    'Chr',
    'Start',
    'Ref',
    'Alt',
    'Gene.refGene',
    'PredictedFunc.refGene',
    'PredConsequence.refGene',
    'avsnp147',
    'Frequencies',
    'Cohort',
    'Allele number',
    'Allele ALT count'
]

# Criando o novo DataFrame apenas com as colunas selecionadas
try:
    df_selecionado = df_bruto[colunas_para_manter].copy()
    print("Colunas selecionadas com sucesso!")
    print(f"Estrutura atual: {df_selecionado.shape[1]} colunas e {df_selecionado.shape[0]} registros.")

except KeyError as e:
    print("\n[ERRO] Alguma coluna não foi encontrada.")
    print(f"Verifique o nome: {e}")

Colunas selecionadas com sucesso!
Estrutura atual: 12 colunas e 2382573 registros.


In [6]:
#colunas com valores ausentes
#gemini
colunas_analise = ['Gene.refGene', 'avsnp147', 'Frequencies']

for coluna in colunas_analise:
    # Garante que a coluna existe no DataFrame selecionado
    if coluna in df_selecionado.columns:
        # Conta nulos do tipo NaN
        qtd_nulos = df_selecionado[coluna].isnull().sum()

        # Conta quantas vezes a string exata "NA" aparece
        qtd_string_na = (df_selecionado[coluna] == 'NA').sum()

        # Total acumulado de ausentes nesta coluna
        total_ausentes = qtd_nulos + qtd_string_na

        print(f"\nColuna: {coluna}")
        print(f"  • Valores nulos/vazios (NaN): {qtd_nulos}")
        print(f"  • Valores com o texto 'NA': {qtd_string_na}")
        print(f"  • Total de registros afetados: {total_ausentes}")
    else:
        print(f"\nA coluna '{coluna}' não foi encontrada no DataFrame.")


Coluna: Gene.refGene
  • Valores nulos/vazios (NaN): 0
  • Valores com o texto 'NA': 0
  • Total de registros afetados: 0

Coluna: avsnp147
  • Valores nulos/vazios (NaN): 669289
  • Valores com o texto 'NA': 0
  • Total de registros afetados: 669289

Coluna: Frequencies
  • Valores nulos/vazios (NaN): 0
  • Valores com o texto 'NA': 0
  • Total de registros afetados: 0


Como só apresentou valores vazios na coluna avsnp147 e ela é muito imoprtante, ela será mantida.

In [7]:
#tratando valores ausentes
# Criando a cópia oficial dos dados selecionados
df_tratado = df_selecionado.copy()

# Vamos apenas tratar a coluna avsnp147
df_tratado['avsnp147'] = df_tratado['avsnp147'].fillna('Nao_catalogado')

print(f"• Total de registros preservados: {len(df_tratado)}")

print(f"• Total de registros preservados: {len(df_tratado)}")
print(f"• Variantes sem rsID mapeadas como 'Nao_catalogado': {sum(df_tratado['avsnp147'] == 'Nao_catalogado')}")

• Total de registros preservados: 2382573
• Total de registros preservados: 2382573
• Variantes sem rsID mapeadas como 'Nao_catalogado': 669289


In [8]:
#(gemini)
print("--- INICIANDO PADRONIZAÇÃO DOS GENES ---")

# Criando uma cópia para o processo de padronização
df_padronizado = df_tratado.copy()

# 1. Garantir que tudo é string e remover espaços extras nas pontas
df_padronizado['Gene.refGene'] = df_padronizado['Gene.refGene'].astype(str).str.strip()

# 2. Padronizar os separadores (se houver vírgula ou barra, transforma em ponto e vírgula)
df_padronizado['Gene.refGene'] = df_padronizado['Gene.refGene'].str.replace(',', ';').str.replace('/', ';')

# Contagem antes da divisão para o relatório
linhas_antes = len(df_padronizado)

# 3. A ESTRATÉGIA: Separar os genes compostos.
# O .str.split(';') transforma "BRCA1;NBR2" em uma lista ['BRCA1', 'NBR2']
df_padronizado['Gene.refGene'] = df_padronizado['Gene.refGene'].str.split(';')

# O .explode() duplica a linha para cada item da lista anterior
df_padronizado = df_padronizado.explode('Gene.refGene')

# 4. Limpeza pós-divisão (remover espaços que possam ter ficado após o ponto e vírgula e colocar em maiúsculo)
df_padronizado['Gene.refGene'] = df_padronizado['Gene.refGene'].str.strip().str.upper()

# Remoção do Gene.refGene = NONE
df_padronizado = df_padronizado[df_padronizado['Gene.refGene'] != 'NONE']


# --- RELATÓRIO DA PADRONIZAÇÃO ---
linhas_depois = len(df_padronizado)
print("\n" + "="*40)
print("  RELATÓRIO DE PADRONIZAÇÃO DE GENES")
print("="*40)
print(f"• Linhas antes da expansão: {linhas_antes}")
print(f"• Linhas após expandir genes compostos: {linhas_depois}")
print(f"• Novas conexões geradas por desmembramento: {linhas_depois - linhas_antes}")
print("\nExemplo dos genes padronizados (Top 5 mais frequentes):")
print(df_padronizado['Gene.refGene'].value_counts().head(30))
print("="*40)

--- INICIANDO PADRONIZAÇÃO DOS GENES ---

  RELATÓRIO DE PADRONIZAÇÃO DE GENES
• Linhas antes da expansão: 2382573
• Linhas após expandir genes compostos: 2439148
• Novas conexões geradas por desmembramento: 56575

Exemplo dos genes padronizados (Top 5 mais frequentes):
Gene.refGene
MIR1268A    2339
MUC4        2219
NBPF20      1948
TTN         1781
MUC16       1685
NBPF25P     1539
CSMD1       1369
SYNE1       1361
OBSCN       1313
NBPF9       1311
NEB         1294
DNAH17      1290
RYR2        1256
RYR3        1198
NBPF1       1160
DNAH11      1155
SSPO        1138
MIR548AZ    1105
RYR1        1099
PKD1L2      1081
MUC5B       1077
PDE4DIP     1060
HLA-DRB1    1023
CFAP46       998
NBPF8        980
MUC2         976
AHNAK2       973
FRAS1        964
ADGRV1       950
CDH23        946
Name: count, dtype: int64


In [9]:
# Criando o identificador único
# (gemini)
df_padronizado['Variant_ID'] = (
    df_padronizado['Chr'].astype(str) + ":" +
    df_padronizado['Start'].astype(str) + ":" +
    df_padronizado['Ref'].astype(str) + ":" +
    df_padronizado['Alt'].astype(str)
)

print("[SUCESSO] Chave primária 'Variant_ID' gerada com sucesso!")

# --- EXIBIÇÃO DE VALIDAÇÃO ---
print("\n" + "="*40)
print("  AMOSTRA DOS NOVOS IDENTIFICADORES (IDs)")
print("="*40)
# Mostra as colunas de origem e o ID gerado para conferência
display(df_padronizado[['Chr', 'Start', 'Ref', 'Alt', 'Variant_ID']].head(15))
print("="*40)

[SUCESSO] Chave primária 'Variant_ID' gerada com sucesso!

  AMOSTRA DOS NOVOS IDENTIFICADORES (IDs)


,Chr,Start,Ref,Alt,Variant_ID
0,1,13116,T,G,1:13116:T:G
1,1,13244,G,A,1:13244:G:A
2,1,13248,C,G,1:13248:C:G
3,1,13273,G,C,1:13273:G:C
4,1,13302,C,T,1:13302:C:T
5,1,13380,C,G,1:13380:C:G
6,1,13417,-,GAGA,1:13417:-:GAGA
7,1,13418,G,A,1:13418:G:A
8,1,13479,A,T,1:13479:A:T
9,1,13494,A,G,1:13494:A:G


In [10]:
#(gemini)
# 1. Converter para Inteiro usando o Pandas
df_padronizado['Start'] = pd.to_numeric(df_padronizado['Start'], errors='coerce').fillna(0).astype('int64')
df_padronizado['Allele number'] = pd.to_numeric(df_padronizado['Allele number'], errors='coerce').fillna(0).astype('int64')
df_padronizado['Allele ALT count'] = pd.to_numeric(df_padronizado['Allele ALT count'], errors='coerce').fillna(0).astype('int64')

# 2. Converter para Decimal (Float)
df_padronizado['Frequencies'] = pd.to_numeric(df_padronizado['Frequencies'], errors='coerce').fillna(0.0).astype('float64')

print("[SUCESSO] Tipos de dados convertidos com sucesso!")

# --- RELATÓRIO DE VALIDAÇÃO DOS TIPOS ---
print("\n" + "="*40)
print("  VERIFICAÇÃO FINAL DOS TIPOS DE DADOS")
print("="*40)
# Filtra para mostrar apenas os tipos das colunas que alteramos
colunas_convertidas = ['Start', 'Allele number', 'Allele ALT count', 'Frequencies']
print(df_padronizado[colunas_convertidas].dtypes)
print("="*40)

[SUCESSO] Tipos de dados convertidos com sucesso!

  VERIFICAÇÃO FINAL DOS TIPOS DE DADOS
Start                 int64
Allele number         int64
Allele ALT count      int64
Frequencies         float64
dtype: object


In [11]:
#Filtragem gemini
linhas_antes_filtro = len(df_padronizado)

# Criando a máscara de filtro: mantém se contiver 'exonic' OU contiver 'splicing' (ignorando maiúsculas/minúsculas)
filtro_mvp = df_padronizado['PredictedFunc.refGene'].str.contains('exonic|splicing', case=False, na=False)

# Aplicando o filtro no DataFrame
df_mvp = df_padronizado[filtro_mvp].copy()

linhas_depois_filtro = len(df_mvp)
linhas_removidas = linhas_antes_filtro - linhas_depois_filtro

# --- RELATÓRIO DE IMPACTO DO FILTRO ---
print("\n" + "="*40)
print("  RELATÓRIO: FILTRAGEM FUNCIONAL DO MVP")
print("="*40)
print(f"• Total de linhas antes do filtro: {linhas_antes_filtro}")
print(f"• Linhas descartadas (baixa relevância): {linhas_removidas}")
print(f"• Linhas preservadas para o Neo4j: {linhas_depois_filtro}")
print(f"• Redução total do volume de dados: {(linhas_removidas / linhas_antes_filtro)*100:.2f}%")
print("\nCategorias funcionais que restaram no seu MVP:")
print(df_mvp['PredictedFunc.refGene'].value_counts())
print("="*40)


  RELATÓRIO: FILTRAGEM FUNCIONAL DO MVP
• Total de linhas antes do filtro: 2439148
• Linhas descartadas (baixa relevância): 2363101
• Linhas preservadas para o Neo4j: 76047
• Redução total do volume de dados: 96.88%

Categorias funcionais que restaram no seu MVP:
PredictedFunc.refGene
ncRNA_exonic             72343
splicing                  3039
ncRNA_splicing             404
exonic;splicing            193
ncRNA_exonic;splicing       68
Name: count, dtype: int64


In [14]:
#eliminando duplicadaas gemini
linhas_antes_duplicadas = len(df_mvp)

colunas_chave = ['Chr', 'Start', 'Ref', 'Alt', 'Gene.refGene']

# Removendo as duplicatas
df_final = df_mvp.drop_duplicates(subset=colunas_chave, keep='first').copy()

linhas_depois_duplicadas = len(df_final)
total_duplicatas_removidas = linhas_antes_duplicadas - linhas_depois_duplicadas



In [15]:
# Listando os nomes de todas as colunas do DataFrame final
print("--- COLUNAS DISPONÍVEIS NO DATAFRAME FINAL ---")
print(df_final.columns.tolist())

--- COLUNAS DISPONÍVEIS NO DATAFRAME FINAL ---
['Chr', 'Start', 'Ref', 'Alt', 'Gene.refGene', 'PredictedFunc.refGene', 'PredConsequence.refGene', 'avsnp147', 'Frequencies', 'Cohort', 'Allele number', 'Allele ALT count', 'Variant_ID']


In [18]:
print("--- APLICANDO PADRONIZAÇÃO DE NOMES PARA O NEO4J ---")

# Dicionário de mapeamento (ajustado para os nomes atuais do seu DataFrame)
mapeamento_colunas = {
    'Chr': 'CHR',  # Caso a coluna Chr ainda esteja aí, já fica em maiúsculo
    'Start': 'START_POSITION',
    'Ref': 'REF_ALLELE',
    'Alt': 'ALT_ALLELE',
    'Gene.refGene': 'GENE_NAME',
    'PredictedFunc.refGene': 'PREDICTED_FUNCTION',
    'PredConsequence.refGene': 'PRED_CONSEQUENCE',
    'avsnp147': 'AVSNP147',
    'Frequencies': 'FREQUENCY',
    'Cohort': 'COHORT_NAME',
    'Allele number': 'ALLELE_NUMBER',
    'Allele ALT count': 'ALLELE_ALT_COUNT',
    'Variant_ID': 'VARIANT_ID'
}



# Renomeando as colunas no DataFrame final
df_final = df_final.rename(columns=mapeamento_colunas)

print("[SUCESSO] Colunas padronizadas para o Neo4j!")
print("\nNovos nomes das colunas:")
print(df_final.columns.tolist())

--- APLICANDO PADRONIZAÇÃO DE NOMES PARA O NEO4J ---
[SUCESSO] Colunas padronizadas para o Neo4j!

Novos nomes das colunas:
['CHR', 'START_POSITION', 'REF_ALLELE', 'ALT_ALLELE', 'GENE_NAME', 'PREDICTED_FUNCTION', 'PRED_CONSEQUENCE', 'AVSNP147', 'FREQUENCY', 'COHORT_NAME', 'ALLELE_NUMBER', 'ALLELE_ALT_COUNT', 'VARIANT_ID']


In [19]:
print("--- CONSTRUÇÃO DAS ENTIDADES (NÓS) ---")

# ==========================================
# 1. CONSTRUÇÃO DA ENTIDADE: GENE
# ==========================================
# Para o nó Gene, precisamos apenas do símbolo único do gene (gene_symbol)
df_nodes_gene = df_final[['GENE_NAME']].copy()

# Renomeando para o padrão solicitado pela tarefa (gene_symbol)
#df_nodes_gene = df_nodes_gene.rename(columns={'GENE_NAME': 'gene_symbol'})

# Remover duplicatas: o gene BRCA1 aparece milhares de vezes na tabela original,
# mas no Neo4j ele precisa existir como um ÚNICO nó.
df_nodes_gene = df_nodes_gene.drop_duplicates().reset_index(drop=True)


# ==========================================
# 2. CONSTRUÇÃO DA ENTIDADE: VARIANT
# ==========================================
# Lista das colunas que pertencem exclusivamente às características da variante
colunas_variante = [
    'VARIANT_ID',
    'CHR',
    'START_POSITION',
    'REF_ALLELE',
    'ALT_ALLELE',
    'FREQUENCY',
    'PRED_CONSEQUENCE',
    'PREDICTED_FUNCTION',
    'COHORT_NAME',
    'AVSNP147'
]

# Garantindo que pegamos apenas as colunas que existem de fato no seu DataFrame atual
colunas_presentes = [col for col in colunas_variante if col in df_final.columns]

df_nodes_variant = df_final[colunas_presentes].copy()



# Remover duplicatas: como uma variante pode ter sido duplicada se ela afetava mais de um gene,
# aqui limpamos para que cada variante tenha apenas uma linha com suas propriedades físicas.
df_nodes_variant = df_nodes_variant.drop_duplicates(subset=['VARIANT_ID']).reset_index(drop=True)


# ==========================================
# RELATÓRIO DE GERAÇÃO DAS ENTIDADES
# ==========================================
print("\n" + "="*40)
print("  RELATÓRIO: ENTIDADES GERADAS COM SUCESSO")
print("="*40)
print(f"• Nós do tipo GENE criados:     {len(df_nodes_gene)}")
print(f"• Nós do tipo VARIANT criados:  {len(df_nodes_variant)}")
print("="*40)

print("\nAmostra da Entidade GENE:")
display(df_nodes_gene.head(3))

print("\nAmostra da Entidade VARIANT:")
display(df_nodes_variant.head(3))

--- CONSTRUÇÃO DAS ENTIDADES (NÓS) ---

  RELATÓRIO: ENTIDADES GERADAS COM SUCESSO
• Nós do tipo GENE criados:     6922
• Nós do tipo VARIANT criados:  74753

Amostra da Entidade GENE:


,GENE_NAME
0,DDX11L1
1,WASH7P
2,MIR6859-1



Amostra da Entidade VARIANT:


,VARIANT_ID,CHR,START_POSITION,REF_ALLELE,ALT_ALLELE,FREQUENCY,PRED_CONSEQUENCE,PREDICTED_FUNCTION,COHORT_NAME,AVSNP147
0,1:13244:G:A,1,13244,G,A,0.002193,NaN,ncRNA_exonic,SABE609,Nao_catalogado
1,1:13248:C:G,1,13248,C,G,0.004149,NaN,ncRNA_exonic,SABE609,Nao_catalogado
2,1:13273:G:C,1,13273,G,C,0.113333,NaN,ncRNA_exonic,SABE609,rs531730856


In [20]:
#gemini
print("---  CONSTRUÇÃO DOS RELACIONAMENTOS - PADRÃO TAREFA 10 ---")

# Criando a tabela de relacionamentos conectando VARIANT_ID diretamente a GENE_NAME
df_rel_ocorre_em = df_final[['VARIANT_ID', 'GENE_NAME']].copy()

# Removendo duplicatas de conexões entre a mesma variante e o mesmo gene
df_rel_ocorre_em = df_rel_ocorre_em.drop_duplicates().reset_index(drop=True)

# --- RELATÓRIO DE GERAÇÃO DOS RELACIONAMENTOS ---
print("\n" + "="*40)
print("  RELATÓRIO: RELACIONAMENTOS GERADOS (OCORRE_EM)")
print("="*40)
print(f"• Total de conexões 'OCORRE_EM' estruturadas: {len(df_rel_ocorre_em)}")
print("="*40)

print("\nAmostra do arquivo de relacionamentos:")
display(df_rel_ocorre_em.head(5))

---  CONSTRUÇÃO DOS RELACIONAMENTOS - PADRÃO TAREFA 10 ---

  RELATÓRIO: RELACIONAMENTOS GERADOS (OCORRE_EM)
• Total de conexões 'OCORRE_EM' estruturadas: 75953

Amostra do arquivo de relacionamentos:


,VARIANT_ID,GENE_NAME
0,1:13244:G:A,DDX11L1
1,1:13248:C:G,DDX11L1
2,1:13273:G:C,DDX11L1
3,1:13302:C:T,DDX11L1
4,1:13380:C:G,DDX11L1


In [21]:
#Geini
print("--- EXPORTAÇÃO DOS ARQUIVOS PARA O NEO4J ---")

# 1. Exportando os Nós de Variantes
df_nodes_variant.to_csv('variants.csv', index=False, encoding='utf-8')
print("[OK] Arquivo 'variants.csv' gerado com sucesso!")

# 2. Exportando os Nós de Genes
df_nodes_gene.to_csv('genes.csv', index=False, encoding='utf-8')
print("[OK] Arquivo 'genes.csv' gerado com sucesso!")

# 3. Exportando os Relacionamentos (Arestas)
df_rel_ocorre_em.to_csv('variant_gene.csv', index=False, encoding='utf-8')
print("[OK] Arquivo 'variant_gene.csv' gerado com sucesso!")

# --- VERIFICAÇÃO DE ARQUIVOS NO DISCO ---
import os

print("\n" + "="*40)
print("  RESUMO DOS ARQUIVOS EXPORTADOS")
print("="*40)
arquivos_esperados = ['variants.csv', 'genes.csv', 'variant_gene.csv']

for arquivo in arquivos_esperados:
    if os.path.exists(arquivo):
        tamanho_mb = os.path.getsize(arquivo) / (1024 * 1024)
        print(f"• {arquivo} -> Criado ({tamanho_mb:.2f} MB)")
    else:
        print(f"• [ERRO] {arquivo} não foi encontrado no disco.")



--- EXPORTAÇÃO DOS ARQUIVOS PARA O NEO4J ---
[OK] Arquivo 'variants.csv' gerado com sucesso!
[OK] Arquivo 'genes.csv' gerado com sucesso!
[OK] Arquivo 'variant_gene.csv' gerado com sucesso!

  RESUMO DOS ARQUIVOS EXPORTADOS
• variants.csv -> Criado (5.48 MB)
• genes.csv -> Criado (0.06 MB)
• variant_gene.csv -> Criado (1.82 MB)
